# Lab 5 – Encoder-only Models for Text Classification

## Task 1: Dataset Preparation

### Step 1 – Load the IMDB dataset and inspect its structure

In [1]:
from datasets import load_dataset

# Load the full IMDB dataset
dataset = load_dataset("imdb")
print(dataset)
print("\nFeatures:", dataset["train"].features)
print("\nFirst training sample:")
print(dataset["train"][0])

c:\repos\aait-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

First training sample:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain politica

### Step 2 – Create train, validation, and test splits

In [2]:
# Sample 1000 random examples from training and 200 from test
train_sample = dataset["train"].shuffle(seed=42).select(range(1000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

# Split the 1000 training samples into 900 train / 100 validation
train_val_split = train_sample.train_test_split(test_size=100, seed=42)
train_dataset = train_val_split["train"]   # 900 samples
val_dataset   = train_val_split["test"]    # 100 samples

print(f"Train size      : {len(train_dataset)}")
print(f"Validation size : {len(val_dataset)}")
print(f"Test size       : {len(test_dataset)}")

Train size      : 900
Validation size : 100
Test size       : 200


### Step 3 – Load the BERT tokenizer with AutoTokenizer

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer class  : {type(tokenizer).__name__}")
print(f"Vocabulary size  : {tokenizer.vocab_size}")
print(f"Max model input  : {tokenizer.model_max_length}")

Tokenizer class  : BertTokenizer
Vocabulary size  : 30522
Max model input  : 512


### Step 4 – Tokenize all dataset splits using `Dataset.map`

In [4]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,          # truncate to model_max_length (512 for BERT)
        max_length=tokenizer.model_max_length,
        padding=False,            # padding will be handled by the data collator later
    )

train_tokenized = train_dataset.map(tokenize, batched=True)
val_tokenized   = val_dataset.map(tokenize, batched=True)
test_tokenized  = test_dataset.map(tokenize, batched=True)

print(train_tokenized)
print("\nColumns after tokenization:", train_tokenized.column_names)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 900
})

Columns after tokenization: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


### Step 5 – Inspect tokenization with `convert_ids_to_tokens`

In [5]:
SAMPLE_IDX = 0  # change to inspect a different sample

sample = train_tokenized[SAMPLE_IDX]
input_ids = sample["input_ids"]

tokens = tokenizer.convert_ids_to_tokens(input_ids)

print(f"Original text (first 200 chars):\n  {sample['text'][:200]}\n")
print(f"Label: {sample['label']}  (0 = negative, 1 = positive)\n")
print(f"Number of tokens : {len(tokens)}")
print(f"Tokens           : {tokens}")

Original text (first 200 chars):
  Little Edie and Big Edie are characters that anyone can feel compassion for. Even though their house was filthy, this is somehow understandable considering their mental illness. On the message board a

Label: 1  (0 = negative, 1 = positive)

Number of tokens : 351
Tokens           : ['[CLS]', 'little', 'ed', '##ie', 'and', 'big', 'ed', '##ie', 'are', 'characters', 'that', 'anyone', 'can', 'feel', 'compassion', 'for', '.', 'even', 'though', 'their', 'house', 'was', 'filthy', ',', 'this', 'is', 'somehow', 'understand', '##able', 'considering', 'their', 'mental', 'illness', '.', 'on', 'the', 'message', 'board', 'a', 'poster', 'wrote', 'that', '"', 'little', 'ed', '##ie', 'has', 'the', 'coping', 'skills', 'of', 'an', 'eight', 'year', 'old', '.', '"', 'this', 'reminded', 'me', 'of', 'when', 'in', 'the', 'drama', '##tized', '2009', 'version', ',', 'big', 'ed', '##ie', 'says', 'to', 'little', 'ed', '##ie', ',', '"', 'if', 'you', "'", 're', 'stuck', ',', 'it'

## Task 2: Model Fine-tuning

### Step 1 – Data collator

In [6]:
import torch
import subprocess

print("PyTorch version         :", torch.__version__)
print("PyTorch CUDA build ver  :", torch.version.cuda)   # None = CPU-only build
print("CUDA available          :", torch.cuda.is_available())

# Check NVIDIA driver
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        print("GPU info                :", result.stdout.strip())
    else:
        print("nvidia-smi error        :", result.stderr.strip())
except FileNotFoundError:
    print("nvidia-smi not found – NVIDIA drivers may not be installed")

PyTorch version         : 2.5.1+cu121
PyTorch CUDA build ver  : 12.1
CUDA available          : True
GPU info                : NVIDIA GeForce RTX 3060 Laptop GPU, 591.74, 6144 MiB


In [7]:
from transformers import DataCollatorWithPadding

# Pads each batch to the length of its longest sequence (more efficient than
# padding everything to model_max_length upfront)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Data collator:", data_collator)

Data collator: DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')


### Step 2 – `compute_metrics` function

In [8]:
import numpy as np
from evaluate import load as load_metric

accuracy_metric = load_metric("accuracy")

def compute_metrics(eval_pred):
    # eval_pred is an EvalPrediction: (logits ndarray (n, c), labels ndarray (n,))
    predictions, labels = eval_pred
    # Convert logits to class ids by taking the argmax over class dimension
    predicted_ids = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predicted_ids, references=labels)

### Step 3 – Define the classification model

In [9]:
from transformers import AutoModelForSequenceClassification

# IMDB is a binary sentiment task: negative=0, positive=1
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
print(model.config)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4451.51it/s]
BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



### Steps 4 & 5 – Training hyperparameters (`TrainingArguments`)

Hyperparameter values are taken from the original BERT paper (Devlin et al., 2018):
- Learning rate: **2e-5** (recommended range 2e-5 – 5e-5)
- Batch size: **16** (paper uses 16 or 32)
- Epochs: **3**
- Weight decay: **0.01**

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-imdb",

    # Evaluation and checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    # Hyperparameters (BERT paper recommendations)
    learning_rate=2e-5,
    per_device_train_batch_size=8,   # small batch → faster per-step on CPU
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,

    # Limit training time: stop after this many steps regardless of epoch count
    max_steps=30,   # CPU-friendly: ~5 min

    # Logging
    logging_steps=10,
    report_to="none",   # disable wandb / tensorboard unless needed
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

### Step 6 & 7 – Create Trainer and start training

In [11]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start fine-tuning
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.507136,0.770000
2,0.563651,0.263779,0.940000
3,0.254418,0.233279,0.920000


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.17s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=100, training_loss=0.4090344715118408, metrics={'train_runtime': 4300.4385, 'train_samples_per_second': 0.744, 'train_steps_per_second': 0.023, 'total_flos': 819393604154880.0, 'train_loss': 0.4090344715118408, 'epoch': 3.4482758620689653})

## Task 3: Model Evaluation

### Step 1 – Evaluate on the test dataset

Because `load_best_model_at_end=True` was set, the trainer already holds the best checkpoint.

In [13]:
from transformers.utils.notebook import NotebookProgressCallback

trainer.remove_callback(NotebookProgressCallback)

test_results = trainer.evaluate(eval_dataset=test_tokenized)
print("Test results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


Test results:
  eval_loss: 0.3038
  eval_accuracy: 0.8650
  eval_runtime: 37.5284
  eval_samples_per_second: 5.3290
  eval_steps_per_second: 0.1870
  epoch: 3.4483


### Step 2 – Sentiment-analysis pipeline

In [14]:
from transformers import pipeline

# Use the fine-tuned model (already loaded in trainer) directly
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=trainer.model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=tokenizer.model_max_length,
)

# Example texts to classify
texts = [
    "This movie was absolutely fantastic! The performances were outstanding.",
    "A complete waste of time. Terrible acting and a nonsensical plot.",
    "It was okay, nothing special but not terrible either.",
    "One of the best films I have ever seen. A true masterpiece!",
    "I fell asleep halfway through. Incredibly boring.",
]

for text in texts:
    result = sentiment_pipeline(text)[0]
    print(f"[{result['label']} | score={result['score']:.4f}]  {text[:80]}")

[POSITIVE | score=0.8596]  This movie was absolutely fantastic! The performances were outstanding.
[NEGATIVE | score=0.9168]  A complete waste of time. Terrible acting and a nonsensical plot.
[NEGATIVE | score=0.6455]  It was okay, nothing special but not terrible either.
[POSITIVE | score=0.8449]  One of the best films I have ever seen. A true masterpiece!
[NEGATIVE | score=0.7704]  I fell asleep halfway through. Incredibly boring.


## Task 4: Experiments

### Experiment 1 – Tuned BERT baseline

Changes vs. baseline:
- **Sequence truncation**: `max_length=128` (vs 512) — attention is $O(n^2)$ in sequence length, so this alone gives ~**16× speedup** in attention layers
- **Slightly larger dataset**: 1 350 train / 150 val / 300 test (vs 900 / 100 / 200)
- **Same step budget**: `max_steps=100`
- **Smaller learning rate**: 1e-5 (vs 2e-5)


In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

# ── Slightly larger splits (but bounded by max_steps) ────────────────────────
dataset_exp = load_dataset("imdb")

exp_train_sample = dataset_exp["train"].shuffle(seed=42).select(range(1500))
exp_test_dataset = dataset_exp["test"].shuffle(seed=42).select(range(100))  # small → fast eval

exp_split   = exp_train_sample.train_test_split(test_size=50, seed=42)  # 50 val samples
exp_train   = exp_split["train"]
exp_val     = exp_split["test"]

print(f"Exp train : {len(exp_train)}")
print(f"Exp val   : {len(exp_val)}   (kept small for fast CPU eval)")
print(f"Exp test  : {len(exp_test_dataset)}  (kept small for fast CPU eval)")

# ── Tokenise (BERT) ───────────────────────────────────────────────────────────
EXP1_MODEL = "google-bert/bert-base-uncased"
exp1_tokenizer = AutoTokenizer.from_pretrained(EXP1_MODEL)

def tokenize_exp1(batch):
    return exp1_tokenizer(batch["text"], truncation=True,
                          max_length=128, padding=False)  # 128 << 512: ~16× faster attention

exp1_train_tok = exp_train.map(tokenize_exp1, batched=True)
exp1_val_tok   = exp_val.map(tokenize_exp1, batched=True)
exp1_test_tok  = exp_test_dataset.map(tokenize_exp1, batched=True)
print("Tokenisation complete.")


c:\repos\aait-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Exp train : 1450
Exp val   : 50   (kept small for fast CPU eval)
Exp test  : 100  (kept small for fast CPU eval)


Map: 100%|██████████| 100/100 [00:00<00:00, 2708.19 examples/s]

Tokenisation complete.


In [ ]:
from transformers import (AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from transformers.utils.notebook import NotebookProgressCallback
import numpy as np
from evaluate import load as load_metric

acc_metric = load_metric("accuracy")

def compute_metrics_exp(eval_pred):
    preds, labels = eval_pred
    return acc_metric.compute(predictions=np.argmax(preds, axis=1), references=labels)

# ── Model ─────────────────────────────────────────────────────────────────────
exp1_model = AutoModelForSequenceClassification.from_pretrained(
    EXP1_MODEL, num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

# ── Training args ─────────────────────────────────────────────────────────────
exp1_args = TrainingArguments(
    output_dir="./bert-imdb-exp1",
    eval_strategy="steps",
    eval_steps=15,
    save_strategy="steps",
    save_steps=15,
    load_best_model_at_end=True,
    learning_rate=1e-5,
    per_device_train_batch_size=8,   # small batch → short forward pass on CPU
    per_device_eval_batch_size=16,
    max_steps=30,                    # ~5 min on CPU
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
)

# ── Trainer ───────────────────────────────────────────────────────────────────
exp1_collator = DataCollatorWithPadding(tokenizer=exp1_tokenizer)

exp1_trainer = Trainer(
    model=exp1_model,
    args=exp1_args,
    train_dataset=exp1_train_tok,
    eval_dataset=exp1_val_tok,
    processing_class=exp1_tokenizer,
    data_collator=exp1_collator,
    compute_metrics=compute_metrics_exp,
)

exp1_trainer.train()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4190.26it/s]
BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

Step,Training Loss,Validation Loss,Accuracy
15,0.733943,0.763825,0.320000
30,0.680913,0.736067,0.420000
45,0.702100,0.703075,0.480000
60,0.670393,0.682362,0.520000
75,0.650970,0.663329,0.560000
90,0.655916,0.653961,0.540000
105,0.642209,0.617026,0.700000
120,0.634727,0.592672,0.720000
135,0.618882,0.579579,0.760000
150,0.620265,0.564115,0.740000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.87it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=200, training_loss=0.647405788898468, metrics={'train_runtime': 55.8615, 'train_samples_per_second': 28.642, 'train_steps_per_second': 3.58, 'total_flos': 104849755560960.0, 'train_loss': 0.647405788898468, 'epoch': 1.098901098901099})

In [5]:
exp1_trainer.remove_callback(NotebookProgressCallback)

exp1_results = exp1_trainer.evaluate(eval_dataset=exp1_test_tok)
print("Experiment 1 – Tuned BERT test results:")
for k, v in exp1_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


Experiment 1 – Tuned BERT test results:
  eval_loss: 0.6005
  eval_accuracy: 0.6700
  eval_runtime: 0.5115
  eval_samples_per_second: 195.5190
  eval_steps_per_second: 13.6860
  epoch: 1.0989


### Experiment 2 – DistilBERT (`distilbert/distilbert-base-uncased`)

DistilBERT is a distilled version of BERT: 40 % fewer parameters, ~2× faster inference and training, while retaining ~97 % of BERT's performance on GLUE benchmarks.

Combined with `max_length=128`, this is the fastest setup in the notebook. Same splits and step budget as Experiment 1.


In [6]:
from transformers import AutoTokenizer

EXP2_MODEL = "distilbert/distilbert-base-uncased"
exp2_tokenizer = AutoTokenizer.from_pretrained(EXP2_MODEL)

def tokenize_exp2(batch):
    return exp2_tokenizer(batch["text"], truncation=True,
                          max_length=128, padding=False)  # 128 tokens – matches Exp 1

exp2_train_tok = exp_train.map(tokenize_exp2, batched=True)
exp2_val_tok   = exp_val.map(tokenize_exp2, batched=True)
exp2_test_tok  = exp_test_dataset.map(tokenize_exp2, batched=True)
print("DistilBERT tokenisation complete.")


c:\repos\aait-lab\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Stefo\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 100/100 [00:00<00:00, 2377.27 examples/s]

DistilBERT tokenisation complete.


In [7]:
from transformers import (AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from transformers.utils.notebook import NotebookProgressCallback

exp2_model = AutoModelForSequenceClassification.from_pretrained(
    EXP2_MODEL, num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

exp2_args = TrainingArguments(
    output_dir="./distilbert-imdb-exp2",
    eval_strategy="steps",
    eval_steps=15,
    save_strategy="steps",
    save_steps=15,
    load_best_model_at_end=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,   # small batch → short forward pass on CPU
    per_device_eval_batch_size=16,
    max_steps=30,                    # ~3 min on CPU (DistilBERT is ~2× faster)
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
)

exp2_collator = DataCollatorWithPadding(tokenizer=exp2_tokenizer)

exp2_trainer = Trainer(
    model=exp2_model,
    args=exp2_args,
    train_dataset=exp2_train_tok,
    eval_dataset=exp2_val_tok,
    processing_class=exp2_tokenizer,
    data_collator=exp2_collator,
    compute_metrics=compute_metrics_exp,
)

exp2_trainer.train()


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9162.27it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss,Validation Loss,Accuracy
15,0.705693,0.704370,0.460000
30,0.657762,0.701137,0.460000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.99it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=30, training_loss=0.6803806781768799, metrics={'train_runtime': 4.6726, 'train_samples_per_second': 51.363, 'train_steps_per_second': 6.42, 'total_flos': 7948043919360.0, 'train_loss': 0.6803806781768799, 'epoch': 0.16483516483516483})

In [ ]:
exp2_trainer.remove_callback(NotebookProgressCallback)

exp2_results = exp2_trainer.evaluate(eval_dataset=exp2_test_tok)
print("Experiment 2 – DistilBERT test results:")
for k, v in exp2_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# ── Side-by-side comparison ───────────────────────────────────────────────────
_baseline = globals().get("test_results", {})
print("\n── Accuracy comparison ──────────────────────────────────────")
print(f"  Baseline BERT (Task 3) : {_baseline.get('eval_accuracy', float('nan')):.4f}")
print(f"  Exp-1 Tuned BERT       : {exp1_results.get('eval_accuracy', float('nan')):.4f}")
print(f"  Exp-2 DistilBERT       : {exp2_results.get('eval_accuracy', float('nan')):.4f}")


Experiment 2 – DistilBERT test results:
  eval_loss: 0.6856
  eval_accuracy: 0.5300
  eval_runtime: 0.2560
  eval_samples_per_second: 390.5530
  eval_steps_per_second: 27.3390
  epoch: 0.1648

── Accuracy comparison ──────────────────────────────────────


NameError: name 'test_results' is not defined